In [2]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [4]:
class network_wake(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, num_layers=2, num_layers_sleep=2):
        super(network_wake, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))

    def forward(self, x, hw=None):
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out)
        
        return out, hw

In [5]:
class network_sleep(nn.Module):
    def __init__(self, input_size, hidden_sleep_size, num_layers=1):
        super(network_sleep, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_sleep_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, len(tokens))

    def forward(self, x, hs=None):
        if hs == None:
            out, hs = self.rnn(x)
        else:
            out, hs = self.rnn(x, hs)

        out = self.sleep_fc(out)
        
        return out, hs

In [6]:
class compressor(nn.Module):
    def __init__(self, input_size, hidden_compressor_size, num_layers=1):
        super(compressor, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_compressor_size, num_layers, nonlinearity='relu', batch_first=True)
        self.compressor_fc = nn.Linear(hidden_compressor_size, 2)

    def forward(self, x, hc=None):
        if hc == None:
            out, hc = self.rnn(x)
        else:
            out, hc = self.rnn(x, hc)

        out = self.compressor_fc(out)
        
        return out, hc

In [7]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [8]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, 3*len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))
        last_visit1 = one_hot_encoded[0]
        last_visit2 = one_hot_encoded[1]
        current_visit = one_hot_encoded[2]
        flag = False
        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,-len(tokens):] = \
                        one_hot_encoded[ii+jj+kk,:]
                    self.X[ii,jj,:7] = \
                        last_visit2.copy()
                    # self.X[ii,jj,7:14] = \
                    #     np.array([0,0,0,0,0,0,1])
                    self.X[ii,jj,7:14] = \
                        last_visit1.copy()
                    # self.X[ii,jj,21:28] = \
                    #     np.array([0,0,0,0,0,0,1])
                    
                    # print(data[ii+jj+kk], data[ii+jj+kk] == 'G')
                    if data[ii+jj+kk] == 'G':
                        last_visit2 = last_visit1.copy()
                        last_visit1 = current_visit.copy()
                        flag = True 
                    # print(data[ii+jj+kk] == 'A' or data[ii+jj+kk] == 'B' or data[ii+jj+kk] == 'C')
                    if flag:
                        if data[ii+jj+kk] == 'A' or data[ii+jj+kk] == 'B' or data[ii+jj+kk] == 'C':
                            # print(data[ii+jj+kk], 'condition1')
                            current_visit = one_hot_encoded[ii+jj+kk,:].copy()
                            flag = False
                            
                        elif data[ii+jj+kk] == 'D' or data[ii+jj+kk] == 'E' or data[ii+jj+kk] == 'F':
                            # print(data[ii+jj+kk], 'condition2')
                            current_visit = one_hot_encoded[ii+jj+kk,:].copy()
                            flag = False 

            # print(last_visit1, last_visit2, current_visit, flag)       
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [9]:
class Dataset_converter_compressor(Dataset):
    def __init__(self, data, mask):
        total_sample = len(data)
        self.X = np.zeros((total_sample-2, len(tokens)))
        self.y = np.zeros((total_sample-2, 2))
        for ii in range(total_sample-2):
            token = data[ii]
            self.X[ii, ord(token)-65] = 1 
            self.y[ii,mask[ii]] = 1
            

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [10]:
### initial training ###
total_samples = 60000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 50
hidden_compressor_size = 5
hidden_sleep_size = 50
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = 3*len(tokens)*working_memory
additional_input = torch.zeros((1,1,4*len(tokens)))

lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members) #
data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = network_wake(input_size, hidden_wake_size, hidden_sleep_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    # X = torch.cat((additional_input, X), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, hw=mem)
        
    loss = criterion(predicted_y[0], y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=2)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 1.9643, accuracy: 0.2260
Iter : 2001, loss: 2.5967, accuracy: 0.3170
Iter : 3001, loss: 1.5048, accuracy: 0.3660
Iter : 4001, loss: 1.6198, accuracy: 0.5620
Iter : 5001, loss: 2.1557, accuracy: 0.6530
Iter : 6001, loss: 2.4448, accuracy: 0.7020
Iter : 7001, loss: 0.9199, accuracy: 0.7370
Iter : 8001, loss: 1.5016, accuracy: 0.7370
Iter : 9001, loss: 1.7963, accuracy: 0.7290
Iter : 10001, loss: 1.1983, accuracy: 0.7430
Iter : 11001, loss: 0.9041, accuracy: 0.7550
Iter : 12001, loss: 1.0438, accuracy: 0.7600
Iter : 13001, loss: 1.6527, accuracy: 0.7550
Iter : 14001, loss: 1.5044, accuracy: 0.7780
Iter : 15001, loss: 0.7767, accuracy: 0.7660
Iter : 16001, loss: 1.5037, accuracy: 0.7810
Iter : 17001, loss: 1.4819, accuracy: 0.7640
Iter : 18001, loss: 3.0691, accuracy: 0.7660
Iter : 19001, loss: 1.3498, accuracy: 0.7730
Iter : 20001, loss: 1.4625, accuracy: 0.7870
Iter : 21001, loss: 1.9457, accuracy: 0.7740
Iter : 22001, loss: 1.5720, accuracy: 0.7970
Iter : 23001, loss:

In [29]:
for ii in range(3):
    print(data_set[20][0][0][ii*7:(ii+1)*7])

print(X)

tensor([0., 0., 0., 0., 1., 0., 0.])
tensor([1., 0., 0., 0., 0., 0., 0.])
tensor([0., 0., 0., 0., 1., 0., 0.])
tensor([[[0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0.]]])


In [25]:
class network_wake2(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, num_layers=2, num_layers_sleep=2):
        super(network_wake2, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(2*len(tokens), len(tokens))
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))

    def forward(self, x, x_, hw=None):
        x = x + self.sleep_fc(x_)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out)
        
        return out, hw

In [26]:
### initial training ###
total_samples = 60000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 50
hidden_compressor_size = 5
hidden_sleep_size = 50
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
additional_input = torch.zeros((1,1,4*len(tokens)))

lr = 6e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members) #
data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = network_wake2(input_size, hidden_wake_size, hidden_sleep_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    # X = torch.cat((additional_input, X), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X[:,:,-7:], X[:,:,:-7])
    else:
        predicted_y, hidden = network1(X[:,:,-7:], X[:,:,:-7], hw=mem)
        
    loss = criterion(predicted_y[0], y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=2)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 2.0121, accuracy: 0.2580
Iter : 2001, loss: 2.9079, accuracy: 0.3650
Iter : 3001, loss: 0.7383, accuracy: 0.5520
Iter : 4001, loss: 1.2967, accuracy: 0.6220
Iter : 5001, loss: 2.9449, accuracy: 0.6510
Iter : 6001, loss: 3.0138, accuracy: 0.6750
Iter : 7001, loss: 0.4789, accuracy: 0.7060
Iter : 8001, loss: 1.3935, accuracy: 0.7020
Iter : 9001, loss: 2.5598, accuracy: 0.7090
Iter : 10001, loss: 1.4952, accuracy: 0.7230
Iter : 11001, loss: 1.0015, accuracy: 0.7560
Iter : 12001, loss: 1.3629, accuracy: 0.7450
Iter : 13001, loss: 1.0945, accuracy: 0.7520
Iter : 14001, loss: 1.5726, accuracy: 0.7450
Iter : 15001, loss: 1.2591, accuracy: 0.7490
Iter : 16001, loss: 1.4899, accuracy: 0.7690
Iter : 17001, loss: 1.0824, accuracy: 0.7620
Iter : 18001, loss: 2.5178, accuracy: 0.7650
Iter : 19001, loss: 1.4697, accuracy: 0.7460
Iter : 20001, loss: 1.4147, accuracy: 0.7700
Iter : 21001, loss: 1.5431, accuracy: 0.7610
Iter : 22001, loss: 1.5335, accuracy: 0.7740
Iter : 23001, loss:

In [42]:
class network_wake3(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, num_layers=2, num_layers_sleep=2):
        super(network_wake3, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(input_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, input_size)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))

    def forward(self, x, x_, hw=None, hs=None):
        # print(x.shape, 'x')
        if hs == None:
            out, hs = self.sleep_rnn(x_)
        else:
            out, hs = self.sleep_rnn(x_, hs)
        # print(out.shape)
        x = x + self.sleep_fc(out)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        out = self.wake_fc(out)
        
        return out, hw, hs

In [54]:
### initial training ###
total_samples = 200000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 50
hidden_compressor_size = 5
hidden_sleep_size = 500
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory

lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members) #
data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = network_wake3(input_size, hidden_wake_size, hidden_sleep_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    # X = torch.cat((additional_input, X), dim=2)
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden_w, hidden_s = network1(X[:,:,-7:], X[:,:,:7])
    else:
        predicted_y, hidden_w, hidden_s = network1(X[:,:,-7:], prev_com, hw=mem, hs=mem_)
        
    loss = criterion(predicted_y[0], y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden_w.clone()
        mem_=hidden_s.clone()
        prev_com = X[:,:,7:14].clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=2)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 2.0698, accuracy: 0.2390
Iter : 2001, loss: 2.2147, accuracy: 0.3840
Iter : 3001, loss: 1.7168, accuracy: 0.4360
Iter : 4001, loss: 2.3575, accuracy: 0.5690
Iter : 5001, loss: 2.1981, accuracy: 0.6370
Iter : 6001, loss: 1.6778, accuracy: 0.6770
Iter : 7001, loss: 1.3987, accuracy: 0.7030
Iter : 8001, loss: 2.2579, accuracy: 0.6900
Iter : 9001, loss: 2.3243, accuracy: 0.7410
Iter : 10001, loss: 1.5574, accuracy: 0.7430
Iter : 11001, loss: 1.3450, accuracy: 0.7370
Iter : 12001, loss: 1.6336, accuracy: 0.7410
Iter : 13001, loss: 0.8368, accuracy: 0.7470
Iter : 14001, loss: 1.9304, accuracy: 0.7370
Iter : 15001, loss: 1.3806, accuracy: 0.7540
Iter : 16001, loss: 2.5439, accuracy: 0.7570
Iter : 17001, loss: 1.4877, accuracy: 0.7450
Iter : 18001, loss: 2.0958, accuracy: 0.7530
Iter : 19001, loss: 2.3105, accuracy: 0.7400
Iter : 20001, loss: 1.1735, accuracy: 0.7520
Iter : 21001, loss: 2.0475, accuracy: 0.7490
Iter : 22001, loss: 1.3802, accuracy: 0.7730
Iter : 23001, loss: